In [5]:
import os
import pandas as pd
from tqdm import tqdm

RAW_DIR = '../data/raw'
OUTPUT_DIR = '../data/yolo_dataset/labels'

os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_and_export_yolo_labels():
    print("Loading datasets...")
    train_df = pd.read_csv(os.path.join(RAW_DIR, 'train.csv'))
    meta_df = pd.read_csv(os.path.join(RAW_DIR, 'train_meta.csv'))
    
    print("Merging data...")
    df = train_df.merge(meta_df, on='image_id', how='left')
    
    unique_images = df['image_id'].unique()
    
    abnormal_df = df[df['class_id'] != 14].copy()
    
    abnormal_df['yolo_class'] = 0
    
    abnormal_df['box_w'] = abnormal_df['x_max'] - abnormal_df['x_min']
    abnormal_df['box_h'] = abnormal_df['y_max'] - abnormal_df['y_min']
    
    abnormal_df['x_center'] = abnormal_df['x_min'] + (abnormal_df['box_w'] / 2)
    abnormal_df['y_center'] = abnormal_df['y_min'] + (abnormal_df['box_h'] / 2)
    
    abnormal_df['x_center'] /= abnormal_df['dim1']
    abnormal_df['y_center'] /= abnormal_df['dim0']
    abnormal_df['w_norm'] = abnormal_df['box_w'] / abnormal_df['dim1']
    abnormal_df['h_norm'] = abnormal_df['box_h'] / abnormal_df['dim0']
    
    cols_to_clip = ['x_center', 'y_center', 'w_norm', 'h_norm']
    abnormal_df[cols_to_clip] = abnormal_df[cols_to_clip].clip(lower=0.0, upper=1.0)
    
    grouped = abnormal_df.groupby('image_id')
    
    print(f"Generating labels for {len(unique_images)} images...")
    for image_id in tqdm(unique_images):
        txt_path = os.path.join(OUTPUT_DIR, f"{image_id}.txt")
        
        if image_id in grouped.groups:
            boxes = grouped.get_group(image_id)[['yolo_class', 'x_center', 'y_center', 'w_norm', 'h_norm']]
            boxes.to_csv(txt_path, sep=' ', header=False, index=False)
        else:
            open(txt_path, 'w').close()
            
    print("Done! All labels are generated successfully.")

process_and_export_yolo_labels()

Loading datasets...
Merging data...
Generating labels for 15000 images...


100%|██████████| 15000/15000 [00:03<00:00, 4404.05it/s]

Done! All labels are generated successfully.
